# How to Use This Notebook

This notebook is designed to be run sequentially from top to bottom without manual intervention. The cells are grouped into numbered steps. Please execute each step in order to run the verification.

- **Step 1: Environment Setup:** Prepares the Kaggle environment, installs dependencies, and verifies TPU access.
- **Step 2: Apply Compatibility Fix:** Downgrades NumPy to prevent known version conflicts.
- **Step 3: Configure Checkpoint Path:** Finds the Llama 3.1 checkpoint dataset and sets the required environment variable.
- **Step 4: Generate Config and Run Verification:** Uses the verified `load_parameters_path` key to generate the complete YAML file and runs the final 1-step training verification. Success is indicated by a "Verification run completed successfully" message.


# Step 1: Environment Setup

This step prepares the Kaggle environment by:
1.  **Verifying JAX and TPU Access:** Ensures the notebook can see the 8 TPU devices.
2.  **Cloning MaxText:** Clones the `google/maxtext` repository, which contains the training scripts.
3.  **Installing Dependencies:** Installs all Python packages required by MaxText from `requirements.txt`.


In [5]:
import os, sys, platform, subprocess

print("Verifying JAX and TPU environment...")
try:
    import jax
    import jax.numpy as jnp
    device_count = jax.device_count()
    print(f"✅ JAX version: {jax.__version__}")
    print(f"✅ Detected {device_count} TPU devices.")
    if device_count != 8:
        print("⚠️ WARNING: Expected 8 TPU devices, but found a different number.")
except Exception as e:
    print(f"❌ ERROR: JAX/TPU verification failed: {e}")
    raise

print("\nCloning MaxText repository...")
if os.path.exists('maxtext'):
    print("✅ 'maxtext' already exists. Skipping clone.")
else:
    subprocess.run(["git", "clone", "https://github.com/google/maxtext.git"], check=True)
    print("✅ MaxText repository cloned.")

# Pin to MaxText commit compatible with JAX 0.4.34 (NVIDIA JAX Release 25.01)
stable_commit_hash = "4651cb3c73de"
print(f"\nChecking out MaxText commit compatible with JAX 0.4.34: {stable_commit_hash}")
try:
    subprocess.run(["git", "checkout", stable_commit_hash], check=True, cwd="maxtext")
    print("✅ Git checkout successful.")
except subprocess.CalledProcessError as e:
    print(f"❌ Git checkout failed: {e}")
    raise

print("\nInstalling dependencies...")
subprocess.run(["apt-get", "update"], check=True, capture_output=True)
subprocess.run(["apt-get", "install", "-y", "pkg-config"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "pip"], check=True, capture_output=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "maxtext/requirements.txt"], check=True, capture_output=True)
print("✅ Dependencies installed.")


Verifying JAX and TPU environment...
✅ JAX version: 0.4.34
✅ Detected 8 TPU devices.

Cloning MaxText repository...
✅ 'maxtext' already exists. Skipping clone.

Checking out MaxText commit compatible with JAX 0.4.34: 4651cb3c73de


HEAD is now at 4651cb3c Merge pull request #1099 from AI-Hypercomputer:mattdavidow-jdi-telemetry


✅ Git checkout successful.

Installing dependencies...
✅ Dependencies installed.


# Step 2: Apply Compatibility Fix

This step downgrades NumPy to version 1.26.4. This is a mandatory step to prevent known compatibility issues between the pre-installed TensorFlow and NumPy 2.x in the Kaggle TPU environment.


In [6]:
import sys
import subprocess

print("Applying NumPy compatibility fix...")
subprocess.run([sys.executable, "-m", "pip", "install", "numpy<2"], check=True, capture_output=True)

print("Verifying NumPy version...")
# We run this in a subprocess to ensure we get the version from the updated environment
result = subprocess.run([sys.executable, "-c", "import numpy as np; print(np.__version__)"], check=True, capture_output=True, text=True)
numpy_version = result.stdout.strip()
print(f"✅ NumPy version is now: {numpy_version}")

if numpy_version != "1.26.4":
    print("⚠️ WARNING: Expected NumPy 1.26.4, but a different version is installed. This may cause issues.")
else:
    print("✅ NumPy version successfully downgraded to 1.26.4.")

print("\nNOTE: A kernel restart may be required for the version change to fully propagate in all contexts.")


Applying NumPy compatibility fix...
Verifying NumPy version...
✅ NumPy version is now: 1.26.4
✅ NumPy version successfully downgraded to 1.26.4.

NOTE: A kernel restart may be required for the version change to fully propagate in all contexts.


# Step 3: Configure Checkpoint Path

This step dynamically locates the pre-converted Llama 3.1 MaxText checkpoint within the attached Kaggle Datasets and sets the `MAXTEXT_CHECKPOINT_DIR` environment variable. This makes the checkpoint path available for the final verification step.


In [7]:
import os
from pathlib import Path

dataset_path = Path("/kaggle/input/llama-3-1-8b-maxtext-checkpoint")

print(f"Inspecting dataset directory: {dataset_path}")
if not dataset_path.exists():
    raise FileNotFoundError(f"Dataset directory not found: {dataset_path}")

required_files = ["_CHECKPOINT_METADATA", "items"]
if all((dataset_path / f).exists() for f in required_files):
    checkpoint_dir = dataset_path
    print(f"✅ Checkpoint found in root directory: {checkpoint_dir}")
else:
    raise FileNotFoundError(f"Could not find required checkpoint files in {dataset_path}")

os.environ["MAXTEXT_CHECKPOINT_DIR"] = str(checkpoint_dir)
print(f"✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR={os.environ['MAXTEXT_CHECKPOINT_DIR']}")


Inspecting dataset directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Checkpoint found in root directory: /kaggle/input/llama-3-1-8b-maxtext-checkpoint
✅ Environment variable set: MAXTEXT_CHECKPOINT_DIR=/kaggle/input/llama-3-1-8b-maxtext-checkpoint


# Step 4: Generate Config and Run Verification

This is the final, fully automated step. It performs the following actions:

1.  **Sets `PYTHONPATH`:** Ensures the MaxText library can be correctly imported.
2.  **Generates YAML:** Creates the `verification_minimal.yml` file using the verified `load_parameters_path` key and the checkpoint path from the previous step.
3.  **Runs Verification:** Executes the MaxText training script as a module (`MaxText.train`) for a single step. 

A successful run will print "✅ Verification run completed successfully." and is the evidence that the entire environment is correctly configured.


In [8]:
import os
import sys
import subprocess
from pathlib import Path

# Set environment variable to resolve TensorFlow protobuf conflict
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# 1. Add maxtext to python path for imports
maxtext_src_path = str(Path.cwd() / "maxtext" / "src")
if maxtext_src_path not in sys.path:
    sys.path.insert(0, maxtext_src_path)
os.environ['PYTHONPATH'] = f"{maxtext_src_path}:{os.environ.get('PYTHONPATH', '')}"
print(f"✅ PYTHONPATH set to include: {maxtext_src_path}")

# 2. Get checkpoint path and define the verified key
checkpoint_path = os.environ.get("MAXTEXT_CHECKPOINT_DIR")
if not checkpoint_path:
    raise ValueError("MAXTEXT_CHECKPOINT_DIR not set. Run the previous step first.")
    
verified_checkpoint_key = "load_parameters_path" # Verified from source code
print(f"✅ Using verified key '{verified_checkpoint_key}' for checkpoint loading.")

# 3. Generate the complete and correct YAML configuration
config_text = f"""
# Auto-generated configuration for verification run
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1
model_name: "llama3.1-8b"
ici_parallelism: -1
mesh_axis_names: ['data', 'fsdp', 'tensor']
{verified_checkpoint_key}: "{checkpoint_path}"
"""

config_path = Path("/kaggle/working/verification_minimal.yml")
config_path.write_text(config_text)
print(f"✅ Wrote config to: {config_path}")
print("--- Config Contents ---")
print(config_text)
print("-----------------------")

# 4. Run the verification script as a module
print("\n🚀 Running 1-step verification...")

cmd = [
    sys.executable, "-m", "MaxText.train", str(config_path)
]

try:
    # Using Popen to stream output in real-time
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, env=os.environ)
    while True:
        output = process.stdout.readline()
        if output == '' and process.poll() is not None:
            break
        if output:
            print(output.strip())
    
    rc = process.poll()
    if rc == 0:
        print("\n✅ Verification run completed successfully.")
    else:
        print(f"\n❌ Verification run failed with return code {rc}")

except FileNotFoundError:
    print("❌ ERROR: Could not find the training module. Is MaxText cloned correctly?")
except Exception as e:
    print(f"❌ An unexpected error occurred: {e}")



✅ PYTHONPATH set to include: /kaggle/working/maxtext/src
✅ Using verified key 'load_parameters_path' for checkpoint loading.
✅ Wrote config to: /kaggle/working/verification_minimal.yml
--- Config Contents ---

# Auto-generated configuration for verification run
run_name: "verification_run_1step"
base_output_directory: "/kaggle/working/maxtext_runs"
steps: 1
per_device_batch_size: 1
model_name: "llama3.1-8b"
ici_parallelism: -1
mesh_axis_names: ['data', 'fsdp', 'tensor']
load_parameters_path: "/kaggle/input/llama-3-1-8b-maxtext-checkpoint"

-----------------------

🚀 Running 1-step verification...
/usr/local/bin/python: Error while finding module specification for 'MaxText.train' (ModuleNotFoundError: No module named 'MaxText')

❌ Verification run failed with return code 1
